<a href="https://colab.research.google.com/github/JediricX/desercionIA/blob/main/Alerta_temprana_abandono_universitario_2_Ingenier%C3%ADa_de_caracter%C3%ADsticas_y_SMOTE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Modelo IA para la detección temprana del abandono universitario basado en dataset OULAD

Objetivo del modelo:

Diseñar e implementar un Sistema de Alerta Temprana basado en Inteligencia Artificial que permita predecir el riesgo de abandono en programas universitarios a distancia, mediante la comparación de algoritmos de clasificación supervisada (Random Forest, Árboles de Decisión, XGBoost y Regresión Logística), con el fin de ofrecer a las instituciones educativas una herramienta preventiva para la retención estudiantil.

Autores:


*   Richard Humberto Campos Ballesteros
*   José Andía



# **1. Importación de librerías**

In [ ]:
# Manejo de datos
import pandas as pd
import numpy as np
import os

# Preprocesamiento
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Modelos
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Métricas
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.metrics import RocCurveDisplay

# Desbalance de clases
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline


# **2. Carga del dataset OULAD**

In [ ]:
# Install dependencies as needed:
#!pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

#Assessments
assessments = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "anlgrbz/student-demographics-online-education-dataoulad",
    "assessments.csv"
)

#Courses
courses = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "anlgrbz/student-demographics-online-education-dataoulad",
    "courses.csv"
)

#student_assessment
student_assessments = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "anlgrbz/student-demographics-online-education-dataoulad",
    "studentAssessment.csv"
)

#student_info
student_info = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "anlgrbz/student-demographics-online-education-dataoulad",
    "studentInfo.csv"
)

#student_registration
student_registration = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "anlgrbz/student-demographics-online-education-dataoulad",
    "studentRegistration.csv"
)

#vle
vle = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "anlgrbz/student-demographics-online-education-dataoulad",
    "vle.csv"
)

#student_vle
student_vle = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "anlgrbz/student-demographics-online-education-dataoulad",
    "studentVle.csv"
)


# **3. Ingeniería de características**

In [ ]:
import pandas as pd
import numpy as np

# --- Ingeniería de Características para Predicción Temprana de Abandono Escolar ---

# --- Base Student Data (pre-merge student_info and student_registration) ---
# Esta es la base común para todas las ventanas temporales
base_student_data = pd.merge(student_info, student_registration,
                             on=['id_student', 'code_module', 'code_presentation'],
                             how='left')

# Determinar la fecha máxima de evaluación para establecer el límite del bucle
max_assessment_date = assessments['date'].max()
print(f"La fecha máxima de evaluación es: {max_assessment_date} días.\n")

# Lista para almacenar los dataframes generados por cada ventana temporal antes de concatenar
all_student_data_list = []

# 2. Definir una ventana temporal para 'predicción temprana' y iterar
current_window_week = 4 # Empezar con 4 semanas

while (current_window_week * 7) <= max_assessment_date:
    window_days = current_window_week * 7
    print(f"\n--- Procesando para la ventana de {current_window_week} semanas ({window_days} días) ---")

    # Crear una copia profunda del DataFrame base para cada iteración
    window_student_data = base_student_data.copy()

    # Añadir la variable objetivo 'dropout'
    window_student_data['dropout'] = window_student_data['final_result'].apply(lambda x: 1 if x == 'Withdrawn' else 0)
    window_student_data = window_student_data.drop('final_result', axis=1)

    # Añadir window_week como característica al DataFrame para esta ventana
    window_student_data['window_week'] = current_window_week

    # --- Características de Evaluaciones Tempranas ---
    student_assessments_details_temp = pd.merge(student_assessments,
                                           assessments[['id_assessment', 'date', 'assessment_type', 'weight', 'code_module', 'code_presentation']],
                                           on='id_assessment',
                                           how='left')

    assessments_data = student_assessments_details_temp[(student_assessments_details_temp['date'] >= (window_days - 28)) & (student_assessments_details_temp['date'] <= window_days)]

    assessments_features = assessments_data.groupby(['id_student', 'code_module', 'code_presentation']).agg(
        num_early_assessments=('id_assessment', 'count'),
        avg_early_score=('score', 'mean'),
        std_early_score=('score', 'std'),
        min_early_score=('score', 'min'),
        max_early_score=('score', 'max'),
        num_failed_early_assessments=('score', lambda x: (x < 40).sum())
    ).reset_index()

    assessments_features['std_early_score'] = assessments_features['std_early_score'].fillna(0)

    window_student_data = pd.merge(window_student_data, assessments_features,
                            on=['id_student', 'code_module', 'code_presentation'],
                            how='left')

    window_student_data['num_early_assessments'] = window_student_data['num_early_assessments'].fillna(0)
    window_student_data['avg_early_score'] = window_student_data['avg_early_score'].fillna(0)
    window_student_data['std_early_score'] = window_student_data['std_early_score'].fillna(0)
    window_student_data['min_early_score'] = window_student_data['min_early_score'].fillna(0)
    window_student_data['max_early_score'] = window_student_data['max_early_score'].fillna(0)
    window_student_data['num_failed_early_assessments'] = window_student_data['num_failed_early_assessments'].fillna(0)
    print("  Características de evaluaciones tempranas agregadas.")

    # --- Características de Actividad VLE Temprana ---
    student_vle_details_temp = pd.merge(student_vle,
                                   vle[['id_site', 'activity_type']],
                                   on='id_site',
                                   how='left')

    vle_data = student_vle_details_temp[(student_vle_details_temp['date'] >= (window_days - 28)) & (student_vle_details_temp['date'] <= window_days)]

    vle_features = vle_data.groupby(['id_student', 'code_module', 'code_presentation']).agg(
        total_early_clicks=('sum_click', 'sum'),
        num_early_vle_interactions=('id_site', 'count'),
        num_unique_early_vle_sites=('id_site', 'nunique'),
        num_early_vle_days=('date', 'nunique')
    ).reset_index()

    vle_features['avg_daily_early_clicks'] = vle_features.apply(lambda row:
        row['total_early_clicks'] / row['num_early_vle_days'] if row['num_early_vle_days'] > 0 else 0,
        axis=1
    )

    vle_activity_type = vle_data.groupby(['id_student', 'code_module', 'code_presentation', 'activity_type']).agg(
        clicks_by_type=('sum_click', 'sum')
    ).unstack(fill_value=0).reset_index()
    vle_activity_type.columns = ['_'.join(col).strip() for col in vle_activity_type.columns.values]
    vle_activity_type.rename(columns={'id_student_': 'id_student', 'code_module_': 'code_module', 'code_presentation_': 'code_presentation'}, inplace=True)

    window_student_data = pd.merge(window_student_data, vle_features,
                            on=['id_student', 'code_module', 'code_presentation'],
                            how='left')
    window_student_data = pd.merge(window_student_data, vle_activity_type,
                            on=['id_student', 'code_module', 'code_presentation'],
                            how='left')

    vle_cols_to_fill = vle_features.columns.drop(['id_student', 'code_module', 'code_presentation']).tolist()
    activity_type_cols_to_fill = [col for col in vle_activity_type.columns if 'clicks_by_type_' in col]

    for col in vle_cols_to_fill + activity_type_cols_to_fill:
        if col in window_student_data.columns:
            window_student_data[col] = window_student_data[col].fillna(0)
    print("  Características de actividad VLE temprana agregadas.")

    # --- Características del Curso (del DataFrame 'courses') ---
    courses_relevant = courses[['code_module', 'code_presentation', 'module_presentation_length']]
    window_student_data = pd.merge(window_student_data, courses_relevant,
                            on=['code_module', 'code_presentation'],
                            how='left')
    print("Longitud de presentación del curso agregada.")

    # Guardar el DataFrame generado para esta ventana en la lista
    all_student_data_list.append(window_student_data)

    # Datos dataframe por ventana
    print(f"  DataFrame generado para la ventana de {current_window_week} semanas guardado.")
    print(f"  Dimensiones del DataFrame generado: {window_student_data.shape}")
    print(f"  Columnas del DataFrame generado: {window_student_data.columns.tolist()[:5]}...")
    pd.set_option('display.max_columns', None)
    print(f" Head: {window_student_data.head(5)}")

    # Incrementar la ventana para la próxima iteración
    current_window_week += 4

# Concatenar todos los DataFrames de la lista en un único DataFrame final
all_student_data_final = pd.concat(all_student_data_list, ignore_index=True)

# Se elimina la característica date_unregistration para evitar data leakage
all_student_data_final.drop(columns=['date_unregistration'], inplace=True)
print("\n--- Resumen del DataFrame final generado ---")
print(f"DataFrame final: Shape {all_student_data_final.shape}, Columnas {all_student_data_final.columns.tolist()[:5]}...")

print("\nValores nulos en el DataFrame final:")
print(all_student_data_final.isnull().sum()[all_student_data_final.isnull().sum() > 0])

## 4. Preprocesamiento de datos: Codificación de variables categóricas

In [ ]:
# Mapeo explícito para variables ordinales
education_mapping = {
    'No Formal quals': 0,
    'Lower Than A Level': 1,
    'A Level or Equivalent': 2,
    'HE Qualification': 3,
    'Post Graduate Qualification': 4
}

imd_mapping = {
    '0-10%': 0,
    '10-20%': 1,
    '20-30%': 2,
    '30-40%': 3,
    '40-50%': 4,
    '50-60%': 5,
    '60-70%': 6,
    '70-80%': 7,
    '80-90%': 8,
    '90-100%': 9
}

age_mapping = {
    '0-35': 0,
    '35-55': 1,
    '55<=': 2
}

# Ahora que tenemos un único DataFrame 'all_student_data_final',
# aplicamos el preprocesamiento directamente a este DataFrame.
print("--- Aplicando preprocesamiento al DataFrame final --- ")

df = all_student_data_final.copy() # Trabajar con una copia para evitar SettingWithCopyWarning

# Imputar 'imd_band' con la moda antes del mapeo, si hay NaNs
# Esto se hace aquí porque `imd_band` es una columna categórica que se mapeará.
if df['imd_band'].isnull().any():
    mode_imd_band = df['imd_band'].mode()[0]
    df['imd_band'] = df['imd_band'].fillna(mode_imd_band)
    print(f"  'imd_band' imputado con la moda: {mode_imd_band}.")

# Aplicar mapeos ordinales
df['highest_education_encoded'] = df['highest_education'].map(education_mapping)
df['imd_band_encoded'] = df['imd_band'].map(imd_mapping)
df['age_band_encoded'] = df['age_band'].map(age_mapping)

# One-hot encoding para variables nominales
# Se elimina region ya que se va aplicar en Perú (implementar pipeline para mantener región?)
# df = pd.get_dummies(df, columns=['gender', 'region'], prefix=['gender', 'region'])
df = pd.get_dummies(df, columns=['gender'], prefix=['gender'])

# Eliminar las columnas originales que han sido codificadas/mapeadas
df = df.drop(columns=['highest_education', 'imd_band', 'age_band'])

# Imputación de Valores Nulos restantes (del paso 5 original)
# Imputar 'date_registration' con la mediana
if df['date_registration'].isnull().any():
    median_date_registration = df['date_registration'].median()
    df['date_registration'] = df['date_registration'].fillna(median_date_registration)
    print(f"  'date_registration' imputado con la mediana: {median_date_registration}.")

# Las columnas 'avg_early_score' y 'total_early_clicks' ya deberían estar rellenas con 0 en el paso de ingeniería de características.
# Si aún quedaran NaNs por alguna razón (ej. por merge posterior), se rellenarían aquí.
if 'avg_early_score' in df.columns and df['avg_early_score'].isnull().any():
    df['avg_early_score'] = df['avg_early_score'].fillna(0)
    print(f"  'avg_early_score' imputado con 0.")

if 'total_early_clicks' in df.columns and df['total_early_clicks'].isnull().any():
    df['total_early_clicks'] = df['total_early_clicks'].fillna(0)
    print(f"  'total_early_clicks' imputado con 0.")

# Imputar los valores nulos restantes para las columnas de actividad VLE y imd_band_encoded con 0
# Estos NaNs pueden surgir de merges o de valores no mapeados en imd_band.
for col in ['clicks_by_type_repeatactivity', 'clicks_by_type_folder', 'imd_band_encoded']:
    if col in df.columns and df[col].isnull().any():
        df[col] = df[col].fillna(0)
        print(f"  '{col}' imputado con 0.")

# Finalmente, guardar el DataFrame procesado en una nueva variable
all_student_data_preprocessed = df

print("\n--- Resumen del DataFrame después de codificación e imputación --- ")
print(f"DataFrame: Shape {all_student_data_preprocessed.shape}")
# Mostrar el conteo de NaNs para verificar
null_counts = all_student_data_preprocessed.isnull().sum()
cols_with_nulls = null_counts[null_counts > 0]
if not cols_with_nulls.empty:
    print(f"  Valores nulos restantes:\n{cols_with_nulls}")
else:
    print(f"  No hay valores nulos restantes.")

**Diccionario de datos**

In [ ]:
print("--- Tipos de datos del DataFrame all_student_data_preprocessed ---")
all_student_data_preprocessed.info()

## 5. Imputación de Valores Nulos

In [ ]:
# Verificar la ausencia de valores nulos en el DataFrame final preprocesado
print("--- Verificación final de valores nulos en el DataFrame preprocesado --- ")

if 'all_student_data_preprocessed' in locals() or 'all_student_data_preprocessed' in globals():
    null_counts_final = all_student_data_preprocessed.isnull().sum()
    cols_with_nulls_final = null_counts_final[null_counts_final > 0]

    if not cols_with_nulls_final.empty:
        print("  Se encontraron valores nulos restantes en el DataFrame final:")
        print(cols_with_nulls_final)
    else:
        print("  No hay valores nulos restantes en el DataFrame 'all_student_data_preprocessed'.")
else:
    print("El DataFrame 'all_student_data_preprocessed' no ha sido creado. Por favor, asegúrese de ejecutar las celdas anteriores.")

## 6. Balanceo de Clases con SMOTE

La variable objetivo `dropout` está desbalanceada (aproximadamente 31% 'Withdrawn' y 69% 'Not Withdrawn'). Para abordar este desbalance, aplicaremos la técnica **SMOTE (Synthetic Minority Over-sampling Technique)**. Esto ayudará a que el modelo no se incline a predecir la clase mayoritaria.

Antes de aplicar SMOTE, necesitamos:
1.  Separar las características (`X`) de la variable objetivo (`y`).
2.  Identificar las columnas numéricas y categóricas para el preprocesamiento, ya que SMOTE opera en datos numéricos. Esto incluye la codificación one-hot de las variables 'object' y la escalada de las variables numéricas.

In [ ]:
print("--- Aplicando separación, preprocesamiento y SMOTE al DataFrame ---")

# Usar el DataFrame completo 'all_student_data_preprocessed'
current_df = all_student_data_preprocessed.copy()

# 1. Separar características (X) y variable objetivo (y)
# 'code_module', 'code_presentation', 'id_student' son identificadores y no deben usarse como features.
# 'date_unregistration' se elimina para evaluar su impacto.
# 'window_week' también se elimina ya que no se fracciona por ventana.
# Se elimina ´region´, se debe determinar si se implementa un pipeline para el tratamiendo de los datos
X_current = current_df.drop(columns=['dropout', 'code_module', 'code_presentation', 'id_student', 'window_week', 'region'])
y_current = current_df['dropout']

# Identificar columnas numéricas, categóricas y booleanas para el ColumnTransformer
# Las columnas booleanas son el resultado de OneHotEncoder previos y ya están en formato numérico (0/1)
numerical_cols = X_current.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X_current.select_dtypes(include='object').columns.tolist()
boolean_cols = X_current.select_dtypes(include='bool').columns.tolist()

# Preprocesamiento: Escalar características numéricas y One-Hot Encode para categóricas
# 'remainder='passthrough'' para mantener las columnas booleanas que ya están en el formato correcto
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols) # Maneja 'disability' y cualquier otro 'object' si lo hubiera
    ],
    remainder='passthrough' # Mantiene las columnas booleanas ya creadas
)

# Primero, dividir los datos en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X_current, y_current, test_size=0.2, random_state=42, stratify=y_current)

print(f"  Dimensiones de X_train antes de SMOTE: {X_train.shape}")
print(f"  Distribución de clases en y_train antes de SMOTE:\n{y_train.value_counts(normalize=True)}")

# Crear un pipeline con preprocesamiento y SMOTE
pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42))
])

# Aplicar el pipeline solo a los datos de entrenamiento
X_train_resampled, y_train_resampled = pipeline.fit_resample(X_train, y_train)

print(f"  Dimensiones de X_train_resampled después de SMOTE: {X_train_resampled.shape}")
print(f"  Distribución de clases en y_train_resampled después de SMOTE:\n{y_train_resampled.value_counts(normalize=True)}")
print(f"  Balanceo de clases completado usando SMOTE.")

# Almacenar los resultados procesados
processed_data_full = {
    'X_train_resampled': X_train_resampled,
    'y_train_resampled': y_train_resampled,
    'X_test': X_test,
    'y_test': y_test,
    'preprocessor': preprocessor
}

print("\n--- Resumen de datos procesados ---")
print(f"  X_train_resampled shape: {processed_data_full['X_train_resampled'].shape}")
print(f"  y_train_resampled distribution:\n{processed_data_full['y_train_resampled'].value_counts(normalize=True)}")
print(f"  X_test shape: {processed_data_full['X_test'].shape}")
print(f"  y_test distribution:\n{processed_data_full['y_test'].value_counts(normalize=True)}")

### 6.1 Visualización del Balanceo de Clases

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("--- Visualizando el balanceo de clases antes y después de SMOTE para el DataFrame completo ---")

# Obtener las distribuciones de clases antes (usando y_test como proxy) y después de SMOTE del DataFrame completo
# `processed_data_full` contiene los datos procesados del DataFrame completo
original_distribution = processed_data_full['y_test'].value_counts(normalize=True).mul(100)
resampled_distribution = processed_data_full['y_train_resampled'].value_counts(normalize=True).mul(100)

# Crear un DataFrame para facilitar la visualización
df_plot = pd.DataFrame({
    'Before SMOTE': original_distribution,
    'After SMOTE': resampled_distribution
}).reset_index()

# Corregir el nombre de la columna para la clase de abandono
df_plot = df_plot.rename(columns={'dropout': 'Dropout Class'})

# Mapear 0 y 1 a etiquetas más descriptivas
df_plot['Dropout Class'] = df_plot['Dropout Class'].map({0: 'Not Withdrawn', 1: 'Withdrawn'})

# Reorganizar el DataFrame para seaborn.barplot
df_plot_melted = df_plot.melt(id_vars='Dropout Class', var_name='Dataset', value_name='Percentage')

plt.figure(figsize=(10, 6))
sns.barplot(x='Dropout Class', y='Percentage', hue='Dataset', data=df_plot_melted, palette='viridis')
plt.title('Distribución de Clases de Abandono (Dropout) Antes y Después de SMOTE (DataFrame Completo)')
plt.xlabel('Clase de Abandono')
plt.ylabel('Porcentaje (%)')
plt.ylim(0, 100)
plt.legend(title='Conjunto de Datos')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

print("--- Visualización de balanceo de clases completada para el DataFrame completo. ---")

## 7. Exportación de Datos Procesados

Para facilitar el uso de los datos preprocesados (`processed_data_full`) en otro notebook o su almacenamiento en un repositorio de GitHub, es necesario guardar sus componentes individualmente. Guardaremos los conjuntos de datos de entrenamiento y prueba (`X_train_resampled`, `y_train_resampled`, `X_test`, `y_test`) como archivos `.npy` y el objeto `preprocessor` (ColumnTransformer) usando `joblib` para preservar su estado.

In [ ]:
import numpy as np
import joblib
import os

from google.colab import drive
drive.mount('/content/drive')

# Guarda los archivos en tu Drive (ejemplo)
output_dir_drive = '/content/drive/MyDrive/Colab Notebooks/Modelos/Data/processed_student_data'
os.makedirs(output_dir_drive, exist_ok=True)

np.save(os.path.join(output_dir_drive, 'X_train_resampled.npy'), processed_data_full['X_train_resampled'])
np.save(os.path.join(output_dir_drive, 'y_train_resampled.npy'), processed_data_full['y_train_resampled'])
np.save(os.path.join(output_dir_drive, 'X_test.npy'), processed_data_full['X_test'])
np.save(os.path.join(output_dir_drive, 'y_test.npy'), processed_data_full['y_test'])
joblib.dump(processed_data_full['preprocessor'], os.path.join(output_dir_drive, 'preprocessor.pkl'))

print("--- Exportación completada. ---")